# Scube Mouse Hypothalamus (2/2): Alignment of consecutive ST slices

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [2]:
import SPACEL
from SPACEL import Scube

In [3]:
seed = 13
from SPACEL.setting import set_environ_seed
set_environ_seed(seed)

Setting environment seed: 13


## Spatial transcriptomic data import

The input data type of Scube is a list of anndata objects for each slice. The anndata objects of each slice must contain a column of spatial domain for alignment in ``.obs``.

In [ ]:
adata = sc.read(f'./{seed}/embryo_E9.5_E16.5_S1_Splane.h5ad')

adata[0].obs

Only considering the two last: ['.5_S1_Splane', '.h5ad'].
Only considering the two last: ['.5_S1_Splane', '.h5ad'].


,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,annotation,Regulon - Arx,Regulon - Bclaf1,Regulon - Bmyc,Regulon - Brf1,Regulon - Cdx2,...,Sclerotome,Smooth muscle,Spinal cord,Submandibular gland,Surface ectoderm,Sympathetic nerve,Thymus,Urogenital ridge,slice_id,spatial_domain
cell_name,,,,,,,,,,,,,,,,,,,,,
147_104,973,6.881411,2203,7.698029,Cavity,0.05467,0.054151,0.026108,0.0,0.026094,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,18


In [5]:
slices = [0,1,2,3,4,5,6,7,]

adata_list=[]

for i in range(8):
    adata_list.append(adata[adata.obs['slice_id'] == slices[i]])
    adata_list[i].obs.spatial_domain = adata_list[i].obs.spatial_domain.astype('category')

In [6]:
adata_list[1]

AnnData object with n_obs × n_vars = 18408 × 23761
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'annotation', 'Regulon - Arx', 'Regulon - Bclaf1', 'Regulon - Bmyc', 'Regulon - Brf1', 'Regulon - Cdx2', 'Regulon - Creb1', 'Regulon - Ctcf', 'Regulon - Cux1', 'Regulon - Dbx1', 'Regulon - Dlx5', 'Regulon - Dlx6', 'Regulon - E2f1', 'Regulon - E2f2', 'Regulon - E2f3', 'Regulon - E2f4', 'Regulon - E2f5', 'Regulon - E2f6', 'Regulon - E2f7', 'Regulon - E2f8', 'Regulon - Elf1', 'Regulon - Elf2', 'Regulon - Elk3', 'Regulon - Emx2', 'Regulon - Esrra', 'Regulon - Ets1', 'Regulon - Ets2', 'Regulon - Etv6', 'Regulon - Evx1', 'Regulon - Ezh2', 'Regulon - Fli1', 'Regulon - Foxa1', 'Regulon - Foxa2', 'Regulon - Foxa3', 'Regulon - Foxc2', 'Regulon - Foxf1', 'Regulon - Foxo1', 'Regulon - Foxp2', 'Regulon - Gabpa', 'Regulon - Gata1', 'Regulon - Gata4', 'Regulon - Gata5', 'Regulon - Gata6', 'Regulon - Gbx2', 'Regulon - Gsx1', 'Regulon - Gtf2f1', 'Regulon - Gt

Plot all raw slices stacked in a single figure. Cells colored by spatial domain.First, we define the color of each spatial domain.

## Pairwise alignment

Here, we perform pairwise alignment of slices. The ``cluster_key`` is a column of ``.obs`` which contain spatial domain information used for alignment. The ``n_neighbors`` affect how much neighbors are considered when calculate similarity of two slices. The ``knn_exclude_cutoff`` affect max distance of neighbors. By default, ``n_neighbors = 15`` and ``knn_exclude_cutoff = n_neighbors`` means a spot/cell in source slices consider 15 neighbors in target slices, and neighbor will be exclude when the distance of neighbors larger than the median of neareast 30 neighbors distance in all spot/cell in target slice. ``p`` affect the exponent of overlap penalty, and a larger p means stronger partial alignment capability.

In [ ]:
Scube.align(adata_list,
      cluster_key='spatial_domain', 
      n_neighbors = 15, 
      n_threads=10,
      p=2,
      seed=seed,
     )

Start alignment...
Alignment slice 1 to 0
Alignment slice 2 to 1
Alignment slice 3 to 2
Alignment slice 4 to 3
Alignment slice 5 to 4
Alignment slice 6 to 5
Alignment slice 7 to 6
Runtime: 2604.8050122261047 s


After alignment, the new spots/cells location coordinates saved in ``.obsm['spatial_aligned']`` of anndata object of each slices. When ``write_loc_path`` provied by users, the location coordinates of all spots/cells of all slices will be saved in ``write_loc_path``.

## LAA & ARI

In [8]:
from sklearn.metrics import pairwise
from sklearn.metrics import adjusted_rand_score

def cal_layer_based_alignment_result_mHypothalamus(alignment, s1, s2):
    #fei added
    labels = []
    labels.extend(s1.obs['annotation'])
    labels.extend(s2.obs['annotation'])

    res = []
    #l_dict = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "6": 6}
    cnt0 = 0
    ir = 0
    max_smallerThan0 = []
    sum_smallerThan0 = []
    min_smallerThan0 = []

    for i, elem in enumerate(alignment):
        #if np.sum(elem)>0:
        if np.max(elem) - np.min(elem) != 0:
            if labels[i] == labels[elem.argmax() + alignment.shape[0]]:
                cnt0 += 1
        else:
            ir += 1
            max_smallerThan0.append(np.max(elem))
            sum_smallerThan0.append(np.sum(elem))
            min_smallerThan0.append(np.min(elem))
    print("the number of sum alignment equals 0:", str(ir))
    if len(sum_smallerThan0)>0:
        print("sum in sum alignment equals 0 is between", np.min(sum_smallerThan0), np.max(sum_smallerThan0))
        print("max in sum alignment equals 0 is between", np.min(max_smallerThan0), np.max(max_smallerThan0))
        print("min in sum alignment equals 0 is between", np.min(min_smallerThan0), np.max(min_smallerThan0))

    res.extend([cnt0])
    return res


def compute_alignment_ari(pi, sliceA, sliceB):
    mapped_clusters = []
    for j in range(pi.shape[1]):
        mapping = pi[:, j]
        #original: if np.sum(mapping) > 0:
        if np.max(mapping) - np.min(mapping) != 0:
            i = np.argmax(mapping)
            mapped_clusters.append(sliceA.obs['annotation'][i])
        else:
            mapped_clusters.append("NULL")
    assert len(sliceB.obs['annotation']) == len(mapped_clusters)
    true_clusters_mapped_region = []
    mapped_clusters_mapped_region = []
    for i in range(len(sliceB.obs['annotation'])):
        if mapped_clusters[i] != "NULL":
            true_clusters_mapped_region.append(sliceB.obs['annotation'][i])
            mapped_clusters_mapped_region.append(mapped_clusters[i])

    ari = adjusted_rand_score(true_clusters_mapped_region, mapped_clusters_mapped_region)
    return ari

In [ ]:
slices = [0,1,2,3,4,5,6,7]

ari_values = []
laa_values = []

for i in range(1, len(slices)):
    target_ind = i-1
    source_ind = i
    f1 = adata_list[target_ind].obsm['spatial_aligned'].to_numpy()
    f2 = adata_list[source_ind].obsm['spatial_aligned'].to_numpy()

    alignment = pairwise.euclidean_distances(f1, f2)
    alignment = np.exp(-alignment)
    
#     name1 = slices[target_ind].split('.')[-1]
#     name2 = slices[source_ind].split('.')[-1]
    # Save as .pkl
#     alignment_df = pd.DataFrame(alignment)
#     alignment_df.to_pickle(f'./{seed}/{name1}_{name2}_AlignmentPi.pkl')
    
    res = cal_layer_based_alignment_result_mHypothalamus(alignment, adata_list[target_ind], adata_list[source_ind])
    laa = res[0]/len(adata_list[source_ind].obs['annotation'])
    ari = compute_alignment_ari(alignment, adata_list[target_ind], adata_list[source_ind])
    ari_values.append(ari)
    laa_values.append(laa)
    print(laa, ari)

# Create DataFrame with desired format
header_row = [f"{slices[i-1]}_{slices[i]}" for i in range(1, len(slices))]
ari_df = pd.DataFrame([ari_values], columns=header_row)
laa_df = pd.DataFrame([laa_values], columns=header_row)

the number of sum alignment equals 0: 0
0.06540634506736202 0.09165384794295171
the number of sum alignment equals 0: 0
0.13321604036648518 0.06531012777735434
the number of sum alignment equals 0: 0
0.1467925630293001 0.08562755948616521
the number of sum alignment equals 0: 0
0.2460804715066758 0.19945950960145623
the number of sum alignment equals 0: 0
0.20590329597440474 0.090451043422034
the number of sum alignment equals 0: 0
0.36288486987207763 0.28136192049306225
the number of sum alignment equals 0: 0
0.31764763852275246 0.2284025704381705


In [ ]:
# Save as a CSV file
ari_df.to_csv(f'./{seed}/embryo_E9.5_E16.5_S1_ARI.csv')
laa_df.to_csv(f'./{seed}/embryo_E9.5_E16.5_S1_LAA.csv')

## Plot alignment results

Save all spots/cells aligned coordiantes and colors in a DataFrame.

In [11]:
c_dict = {'AGM': '#1f77b4',
 'Adipose tissue': '#aec7e8',
 'Adrenal gland': '#ff7f0e',
 'Blood vessel': '#ffbb78',
 'Bone': '#2ca02c',
 'Brain': '#98df8a',
 'Branchial arch': '#d62728',
 'Cartilage': '#ff9896',
 'Cartilage primordium': '#9467bd',
 'Cavity': '#c5b0d5',
 'Choroid plexus': '#8c564b',
 'Connective tissue': '#c49c94',
 'Dermomyotome': '#e377c2',
 'Dorsal root ganglion': '#f7b6d2',
 'Epidermis': '#7f7f7f',
 'Facial nerve': '#c7c7c7',
 'GI tract': '#bcbd22',
 'Head mesenchyme': '#dbdb8d',
 'Heart': '#17becf',
 'Inner ear': '#9edae5',
 'Jaw and tooth': '#393b79',
 'Kidney': '#5254a3',
 'Liver': '#6b6ecf',
 'Lung': '#9c9ede',
 'Lung primordium': '#637939',
 'Meninges': '#8ca252',
 'Mesenchyme': '#b5cf6b',
 'Mesentery': '#cedb9c',
 'Mesothelium': '#8c6d31',
 'Mucosal epithelium': '#bd9e39',
 'Muscle': '#e7ba52',
 'Neural crest': '#e7cb94',
 'Notochord': '#843c39',
 'Olfactory epithelium': '#ad494a',
 'Ovary': '#d6616b',
 'Pancreas': '#e7969c',
 'Sclerotome': '#7b4173',
 'Smooth muscle': '#a55194',
 'Spinal cord': '#ce6dbd',
 'Submandibular gland': '#de9ed6',
 'Surface ectoderm': '#3182bd',
 'Sympathetic nerve': '#6baed6',
 'Thymus': '#9ecae1',
 'Urogenital ridge': '#c6dbef'}

In [ ]:
coo = pd.DataFrame()

for i in range(len(adata_list)):
    loc = adata_list[i].obsm['spatial_aligned'].copy()
    loc['Z'] = i
    loc['slice'] = adata_list[i].obs['slice_id'] + 1
    loc['original_clusters'] = adata_list[i].obs['annotation']
    loc['celltype_colors'] = adata_list[i].obs['annotation'].replace(c_dict)
    coo = pd.concat([coo,loc],axis=0)
    
coo.to_csv(f'./{seed}/SPACEL_embryo_E9.5_E16.5_S1_aligned_coordinates_colors.csv')

In [13]:
coo

,X,Y,Z,slice,original_clusters,celltype_colors
cell_name,,,,,,
147_104,-14.000000,52.000000,0,1,Cavity,#c5b0d5
147_105,-13.000000,52.000000,0,1,Cavity,#c5b0d5
147_106,-12.000000,52.000000,0,1,Cavity,#c5b0d5
147_107,-11.000000,52.000000,0,1,Cavity,#c5b0d5
147_108,-10.000000,52.000000,0,1,Mesenchyme,#b5cf6b
...,...,...,...,...,...,...
700_236,15.967072,-179.605011,7,8,Cavity,#c5b0d5
700_237,16.966700,-179.578415,7,8,Cavity,#c5b0d5
700_238,17.966360,-179.551834,7,8,Cavity,#c5b0d5


In [ ]:
def plot_3d(
    loc,
    val=None,
    color=None,
    figsize=(12,12),
    return_fig=True,
    elev=None,
    azim=None,
    roll=None,
    xlim=None,
    ylim=None,
    zlim=None,
    frameon=True,
    save_path=None,
    save_dpi=150,
    show=True,
    *args,
    **kwargs
):
    """Plot all slices stacked in 3D

    Plot all slices stacked with a given number of subplots rows and columns. Spots/cells colored by spatial domain.

    Args:
        loc: An array of the coordinates of each spots/cells in all slices, which the first three columns are X-axis, Y-axis, Z-axis coordinates.
        val: The colors of each spots/cells given in loc use to plot.
        color: The colors of each spots/cells given in loc use to plot.
        figsize: Size of the figure.
        return_fig: Whether to return the figure.
        elev: The elevation angle in the vertical plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        azim: The azimuth angle in the horizontal plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        xlim: A tuple given the left and right xlims in X-axis.
        ylim: A tuple given the left and right xlims in Y-axis.
        zlim: A tuple given the left and right xlims in Z-axis.
        frameon: Whether to hide the coordinate axes.
        save_path: A string representing the path directory where the figure saved.
        save_dpi: The resolution in dots per inch.
        show: Whether to show the figure.

    Returns:
        A ``matplotlib.figure.Figure`` object
    """
    if 'marker' not in kwargs.keys():
        kwargs['marker'] = 'o'
    if 's' not in kwargs.keys():
        kwargs['s'] = 5
    if 'cmap' not in kwargs.keys():
        kwargs['cmap'] = 'Spectral_r'
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(projection='3d')
    if (elev is not None) and (azim is not None) and (roll is not None):
        ax.view_init(elev, azim, roll)
    if color is None:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=val,*args,**kwargs)
    else:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=color,*args,**kwargs)
    
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set_box_aspect([2, 2, 3])
    if not frameon:
        plt.axis('off')
        plt.grid(b=None)
    if save_path is not None:
        print(save_path)
        plt.savefig(save_path,dpi=save_dpi,bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close()
    if return_fig:
        return fig

In [ ]:
seed = 2024
df = pd.read_csv(f'./{seed}/SPACEL_embryo_E9.5_E16.5_S1_aligned_coordinates_colors.csv')
coords = df[['X', 'Y', 'Z']]

In [ ]:
fig = plot_3d(coords.values, val=None, color=df['celltype_colors'], elev=320, azim=0, roll=0, frameon=False, show=True, s=15, save_dpi=300, return_fig=True) # Scube is from SPACEL

fig.savefig(f'./{seed}/SPACEL_gtAndScube_lymph_3dviz_angle320_0_0_s=15.png', bbox_inches="tight")